# Generator Fixed vs. Variable Cost EDA

Analysis of fixed (startup) and variable (production) costs for thermal generators in the RTS-GMLC 2020-01-27 instance.

---

## Archetype: Baseload Unit

A unit committed for the **entire 48-period horizon** (one commitment per one-hour period).  
Piecewise production costs are in $/hr; since each period = 1 hour, cost per period = $/hr directly, and dispatch in MW × 1 hr = MWh per period.

### Fixed cost levels

| Level | Description |
|---|---|
| **None** | Unit already on — no startup cost, no ramp |
| **Hot** | Cheapest startup tier (shortest min-offline required) |
| **Warm** | Middle tier — only if unit has ≥ 2 startup tiers |
| **Cold** | Most expensive tier (longest min-offline required) |

- 1 startup tier in JSON → Hot=N/A, Warm=N/A, Cold only  
- 2 startup tiers → Hot=N/A, Warm + Cold available  
- 3 startup tiers → Hot + Warm + Cold all available

### Variable cost levels

| Level | Description |
|---|---|
| **Min MC** | Dispatch at the upper MW bound of the cheapest marginal cost segment |
| **Max Power** | Dispatch at pmax for the entire horizon |

For startup cases, the unit ramps from 0 → `ramp_startup_limit` → target via `ramp_up_limit` per period.  
For *None* (already on), dispatch is immediately at target — no ramp.

### Metrics tracked per case

- **Startup cost ($)** — fixed cost for that tier  
- **Variable cost ($)** — total production cost over all 48 periods  
- **Gross cost ($)** — startup + variable  
- **Energy (MWh)** — total MWh produced (dispatch MW × 1 hr/period × 48 periods)  
- **Cost / MWh ($/MWh)** — gross cost ÷ energy  
- **% Fixed** — startup cost as % of gross  
- **% Variable** — production cost as % of gross

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)

## 1. Load Data

In [2]:
DATA_PATH = Path("data/2020-01-27/thermal_generators.json")

# Time horizon constants
N_PERIODS  = 48      # 48 one-hour periods
PERIOD_HRS = 1.0     # hours per period (piecewise costs are in $/hr = $/period)

with open(DATA_PATH) as f:
    thermal_gens = json.load(f)

print(f"Loaded {len(thermal_gens)} thermal generators")
print()

# Count by fuel category and startup tier count
from collections import Counter
cat_counts  = Counter(g["fuel_category"] for g in thermal_gens.values())
tier_counts = Counter(len(g.get("startup", [])) for g in thermal_gens.values())

print("By fuel category:")
for cat, n in sorted(cat_counts.items()):
    print(f"  {cat:<12}: {n}")

print()
print("By startup tier count:")
for n_tiers, count in sorted(tier_counts.items()):
    capability = {1: "cold only", 2: "warm + cold", 3: "hot + warm + cold"}.get(n_tiers, str(n_tiers))
    print(f"  {n_tiers} tier(s) ({capability:<22}): {count} units")

Loaded 73 thermal generators

By fuel category:
  Coal        : 16
  Gas CC      : 10
  Gas CT      : 27
  Nuclear     : 1
  Oil CT      : 12
  Oil ST      : 7

By startup tier count:
  1 tier(s) (cold only             ): 50 units
  2 tier(s) (warm + cold           ): 2 units
  3 tier(s) (hot + warm + cold     ): 21 units


In [3]:
# Quick overview of generator characteristics by category
overview_rows = []
for name, gen in thermal_gens.items():
    overview_rows.append({
        "name":          name,
        "category":      gen.get("fuel_category", "Unknown"),
        "pmin":          gen["power_output_minimum"],
        "pmax":          gen["power_output_maximum"],
        "ramp_up": gen.get("ramp_up_limit", 0),
        "ramp_startup":  gen.get("ramp_startup_limit", gen["power_output_minimum"]),
        "min_up":        gen.get("time_up_minimum", 0),
        "min_down":      gen.get("time_down_minimum", 0),
        "n_startup_tiers": len(gen.get("startup", [])),
    })

df_overview = pd.DataFrame(overview_rows).set_index("name")

print("Per-category averages:")
display(
    df_overview.groupby("category")[["pmin", "pmax", "ramp_up", "min_up", "min_down", "n_startup_tiers"]]
    .mean()
    .round(1)
)

Per-category averages:


,pmin,pmax,ramp_up,min_up,min_down,n_startup_tiers
category,,,,,,
Coal,57.80,144.80,161.20,10.00,11.20,2.90
Gas CC,170.00,355.00,248.40,8.00,5.00,1.00
Gas CT,22.00,55.00,222.00,3.00,3.00,1.00
Nuclear,396.00,400.00,"1,200.00",24.00,48.00,1.00
Oil CT,8.00,20.00,180.00,1.00,1.00,1.00
Oil ST,5.00,12.00,60.00,4.00,2.00,3.00


## 2. Helper Functions

In [4]:
def get_startup_tiers(gen: dict) -> dict:
    """
    Parse startup tiers from the generator dict.
    Returns {'hot': tier|None, 'warm': tier|None, 'cold': tier|None}
    Each tier is {'lag': int, 'cost': float} or None if not available.

    Tiers in the JSON are ordered ascending by lag:
      1 tier  -> [cold]            (hot=None, warm=None)
      2 tiers -> [warm, cold]      (hot=None)
      3 tiers -> [hot, warm, cold]
    """
    tiers = gen.get("startup", [])
    n = len(tiers)
    return {
        "hot":  dict(tiers[0])  if n >= 3 else None,
        "warm": dict(tiers[-2]) if n >= 2 else None,
        "cold": dict(tiers[-1]) if n >= 1 else None,
    }


def find_min_mc_segment(gen: dict) -> dict:
    """
    Find the piecewise production segment with the lowest marginal cost.

    piecewise_production costs are in $/hr.  The marginal cost of a segment
    is (delta_cost_$/hr) / (delta_mw) = $/MWh for the increment.

    Returns:
      mc_per_mwh  : marginal cost of the cheapest segment ($/MWh)
      target_mw   : upper MW bound of the cheapest segment (dispatch target)
      segment_idx : 1-based index of the cheapest segment
    """
    pp = gen["piecewise_production"]
    if len(pp) < 2:
        return {"mc_per_mwh": 0.0, "target_mw": float(pp[0]["mw"]), "segment_idx": 1}

    best_mc  = float("inf")
    best_mw  = float(pp[1]["mw"])
    best_seg = 1

    for i in range(1, len(pp)):
        mw_delta   = pp[i]["mw"]   - pp[i - 1]["mw"]
        cost_delta = pp[i]["cost"] - pp[i - 1]["cost"]
        mc = cost_delta / mw_delta   # $/hr per MW  =  $/MWh
        if mc < best_mc:
            best_mc  = mc
            best_mw  = float(pp[i]["mw"])
            best_seg = i

    return {"mc_per_mwh": best_mc, "target_mw": best_mw, "segment_idx": best_seg}


def piecewise_cost_per_hr(gen: dict, mw: float) -> float:
    """
    Evaluate the piecewise production cost ($/hr) at a given dispatch level (MW).

    Linearly interpolates between breakpoints.  For dispatch below pmin,
    linearly interpolates from (0 MW, $0/hr) to (pmin, piecewise[0][cost]).
    """
    pp = gen["piecewise_production"]
    pmin_mw   = float(pp[0]["mw"])
    pmin_cost = float(pp[0]["cost"])

    if mw <= 0.0:
        return 0.0
    if mw <= pmin_mw:
        return (mw / pmin_mw) * pmin_cost

    for i in range(1, len(pp)):
        if mw <= pp[i]["mw"]:
            frac = (mw - pp[i - 1]["mw"]) / (pp[i]["mw"] - pp[i - 1]["mw"])
            return pp[i - 1]["cost"] + frac * (pp[i]["cost"] - pp[i - 1]["cost"])

    return float(pp[-1]["cost"])   # at or above pmax


def build_dispatch_profile(
    gen: dict,
    target_mw: float,
    has_startup: bool,
    n_periods: int = N_PERIODS,
) -> list[float]:
    """
    Build the per-period dispatch sequence (MW) for a baseload commitment.

    No startup  -> constant at target_mw for all periods.
    With startup:
      Period 0 : min(target, ramp_startup_limit)
      Period t : min(target, prev + ramp_up_limit)
    """
    if not has_startup:
        return [target_mw] * n_periods

    pmin       = float(gen["power_output_minimum"])
    ramp_start = float(gen.get("ramp_startup_limit", pmin))
    ramp_up    = float(gen.get("ramp_up_limit", target_mw))

    profile = []
    current = 0.0
    for t in range(n_periods):
        if t == 0:
            new = min(target_mw, ramp_start)
        else:
            new = min(target_mw, current + ramp_up)
        profile.append(new)
        current = new
    return profile


def compute_case(
    gen: dict,
    startup_cost: float,
    target_mw: float,
    has_startup: bool,
    n_periods: int = N_PERIODS,
    period_hrs: float = PERIOD_HRS,
) -> dict:
    """
    Compute all cost metrics for one (fixed level x variable level) case.

    Returns a dict with: startup_cost, variable_cost, gross_cost, energy_mwh,
    cost_per_mwh, pct_fixed, pct_variable, periods_to_target.
    """
    profile = build_dispatch_profile(gen, target_mw, has_startup, n_periods)

    # Cost per period = ($/hr) * period_hrs;  sum over all periods
    variable_cost = sum(piecewise_cost_per_hr(gen, mw) * period_hrs for mw in profile)
    energy_mwh    = sum(mw * period_hrs for mw in profile)
    gross_cost    = startup_cost + variable_cost

    periods_to_target = next(
        (t for t, mw in enumerate(profile) if mw >= target_mw - 1e-6),
        n_periods - 1,
    )

    return {
        "startup_cost":      startup_cost,
        "variable_cost":     variable_cost,
        "gross_cost":        gross_cost,
        "energy_mwh":        energy_mwh,
        "cost_per_mwh":      gross_cost / energy_mwh if energy_mwh > 0 else float("nan"),
        "pct_fixed":         startup_cost / gross_cost * 100 if gross_cost > 0 else 0.0,
        "pct_variable":      variable_cost / gross_cost * 100 if gross_cost > 0 else 100.0,
        "periods_to_target": periods_to_target,
    }


# Quick self-test with 322_CT_5
_g = thermal_gens["322_CT_5"]
_tiers = get_startup_tiers(_g)
_mc    = find_min_mc_segment(_g)
print(f"322_CT_5 startup tiers : {_tiers}")
print(f"322_CT_5 min MC segment: mc={_mc['mc_per_mwh']:.4f} $/MWh  target={_mc['target_mw']} MW  (seg {_mc['segment_idx']})")
print(f"Piecewise cost at 33 MW: ${piecewise_cost_per_hr(_g, 33.0):.2f}/hr")
print(f"Piecewise cost at 55 MW: ${piecewise_cost_per_hr(_g, 55.0):.2f}/hr")
print()
_case_cold_minmc = compute_case(_g, _tiers["cold"]["cost"], _mc["target_mw"], has_startup=True)
print(f"Cold + MinMC: gross=${_case_cold_minmc['gross_cost']:,.2f}  {_case_cold_minmc['energy_mwh']:.1f} MWh  "
      f"${_case_cold_minmc['cost_per_mwh']:.2f}/MWh  fixed={_case_cold_minmc['pct_fixed']:.1f}%")

322_CT_5 startup tiers : {'hot': None, 'warm': None, 'cold': {'lag': 3, 'cost': 5665.23}}
322_CT_5 min MC segment: mc=23.3345 $/MWh  target=33.0 MW  (seg 1)
Piecewise cost at 33 MW: $1288.38/hr
Piecewise cost at 55 MW: $1886.72/hr

Cold + MinMC: gross=$67,250.79  1573.0 MWh  $42.75/MWh  fixed=8.4%


## 3. Compute Baseload Cases for All Generators

In [5]:
FIXED_LEVELS = ["none", "hot", "warm", "cold"]
VAR_LEVELS   = ["min_mc", "max_power"]

records = []

for name, gen in thermal_gens.items():
    tiers   = get_startup_tiers(gen)
    mc_info = find_min_mc_segment(gen)
    pmax    = float(gen["power_output_maximum"])

    startup_costs = {
        "none": 0.0,
        "hot":  tiers["hot"]["cost"]  if tiers["hot"]  is not None else None,
        "warm": tiers["warm"]["cost"] if tiers["warm"] is not None else None,
        "cold": tiers["cold"]["cost"] if tiers["cold"] is not None else None,
    }
    startup_lags = {
        "none": 0,
        "hot":  tiers["hot"]["lag"]   if tiers["hot"]  is not None else None,
        "warm": tiers["warm"]["lag"]  if tiers["warm"] is not None else None,
        "cold": tiers["cold"]["lag"]  if tiers["cold"] is not None else None,
    }

    targets = {
        "min_mc":    mc_info["target_mw"],
        "max_power": pmax,
    }

    for fixed_level in FIXED_LEVELS:
        sc = startup_costs[fixed_level]
        has_startup = fixed_level != "none"

        for var_level in VAR_LEVELS:
            target = targets[var_level]

            base = {
                "name":           name,
                "fuel_category":  gen.get("fuel_category", "Unknown"),
                "pmin":           float(gen["power_output_minimum"]),
                "pmax":           pmax,
                "fixed_level":    fixed_level,
                "var_level":      var_level,
                "startup_lag":    startup_lags[fixed_level],
                "min_mc_per_mwh": mc_info["mc_per_mwh"],
                "min_mc_target_mw": mc_info["target_mw"],
            }

            if sc is None:
                base["applicable"] = False
                base.update({
                    k: None for k in
                    ["startup_cost", "variable_cost", "gross_cost",
                     "energy_mwh", "cost_per_mwh", "pct_fixed", "pct_variable",
                     "periods_to_target"]
                })
            else:
                metrics = compute_case(gen, sc, target, has_startup)
                base["applicable"] = True
                base.update(metrics)

            records.append(base)

df = pd.DataFrame(records)

n_applicable = df["applicable"].sum()
n_na         = (~df["applicable"]).sum()
print(f"Total records   : {len(df)}")
print(f"Applicable cases: {n_applicable}")
print(f"N/A cases       : {n_na}  (units lacking hot/warm start tiers)")
print()

df_valid = df[df["applicable"]].copy()

Total records   : 584
Applicable cases: 380
N/A cases       : 204  (units lacking hot/warm start tiers)



## 4. Results Summary Table

Cost per MWh ($/MWh) for each generator across all applicable cases.  
Columns: `fixed_level_var_level` (e.g. `none_max_power`, `cold_min_mc`).

In [6]:
# ── Pivot: one row per generator, columns = (fixed_level, var_level), value = $/MWh
CASE_ORDER = [
    ("none", "min_mc"),
    ("none", "max_power"),
    ("hot",  "min_mc"),
    ("hot",  "max_power"),
    ("warm", "min_mc"),
    ("warm", "max_power"),
    ("cold", "min_mc"),
    ("cold", "max_power"),
]

pivot = df.pivot_table(
    index=["name", "fuel_category", "pmin", "pmax", "min_mc_per_mwh", "min_mc_target_mw"],
    columns=["fixed_level", "var_level"],
    values="cost_per_mwh",
    aggfunc="first",
)

# Reorder columns
pivot = pivot.reindex(columns=pd.MultiIndex.from_tuples(CASE_ORDER), level=None)
pivot.columns = [f"{fl}_{vl}" for fl, vl in pivot.columns]
pivot = pivot.reset_index().sort_values(["fuel_category", "name"]).set_index(["name", "fuel_category"])

# Style: green = cheap, red = expensive; NaN cells = gray
cost_cols = [c for c in pivot.columns if "_" in c and any(fl in c for fl in ["none", "hot", "warm", "cold"])]

styled = (
    pivot.style
    .format({
        "pmin":             "{:,.1f} MW",
        "pmax":             "{:,.1f} MW",
        "min_mc_per_mwh":   "${:,.2f}/MWh",
        "min_mc_target_mw": "{:,.1f} MW",
        **{c: lambda x: f"${x:,.2f}" if pd.notna(x) else "N/A" for c in cost_cols},
    })
    .background_gradient(
        subset=cost_cols,
        cmap="RdYlGn_r",   # red = expensive, green = cheap
        axis=None,
    )
    .set_caption("Baseload Archetype — Cost per MWh ($/MWh) by Case")
)

display(styled)

,,pmin,pmax,min_mc_per_mwh,min_mc_target_mw,none_min_mc,none_max_power,hot_min_mc,hot_max_power,warm_min_mc,warm_max_power,cold_min_mc,cold_max_power
name,fuel_category,,,,,,,,,,,,
101_STEAM_3,Coal,30.0 MW,76.0 MW,$14.19/MWh,45.3 MW,$23.36,$21.01,$26.74,$23.05,$28.19,$23.92,$28.60,$24.17
101_STEAM_4,Coal,30.0 MW,76.0 MW,$14.19/MWh,45.3 MW,$23.36,$21.01,$26.74,$23.05,$28.19,$23.92,$28.60,$24.17
102_STEAM_3,Coal,30.0 MW,76.0 MW,$18.46/MWh,45.3 MW,$22.46,$22.15,$25.80,$24.15,$27.25,$25.02,$27.66,$25.27
102_STEAM_4,Coal,30.0 MW,76.0 MW,$18.46/MWh,45.3 MW,$22.46,$22.15,$25.80,$24.15,$27.25,$25.02,$27.66,$25.27
115_STEAM_3,Coal,62.0 MW,155.0 MW,$20.40/MWh,93.0 MW,$22.93,$23.67,$26.24,$25.65,$26.50,$25.81,$28.09,$26.77
116_STEAM_1,Coal,62.0 MW,155.0 MW,$19.69/MWh,93.0 MW,$25.22,$24.20,$28.54,$26.22,$28.80,$26.37,$30.40,$27.33
123_STEAM_2,Coal,62.0 MW,155.0 MW,$19.43/MWh,93.0 MW,$21.93,$24.36,$25.24,$26.33,$25.50,$26.49,$27.09,$27.45
123_STEAM_3,Coal,140.0 MW,350.0 MW,$19.98/MWh,210.0 MW,$23.72,$23.25,N/A,N/A,$25.88,$24.56,$27.42,$25.49
201_STEAM_3,Coal,30.0 MW,76.0 MW,$22.19/MWh,45.3 MW,$25.68,$25.24,$29.01,$27.24,$30.46,$28.11,$30.87,$28.36


---

## 2. Peaking Unit Archetype

A unit that **starts up for peak load hours then shuts back down**. No "already on" state — every commitment incurs a startup cost.

### Fixed cost levels

| Level | Description |
|---|---|
| **Hot** | Cheapest startup tier — available only if unit has 3 startup tiers |
| **Warm** | Middle tier — available if unit has 2 or 3 tiers |
| **Cold** | Most expensive tier — always available |

### Variable cost levels (3 dispatch profiles)

| Level | Target dispatch | Description |
|---|---|---|
| **Max Power** | pmax | Ramp up to pmax, hold if needed, ramp back down |
| **Min MC** | upper bound of cheapest MC segment | Ramp to the most efficient operating point, hold, ramp down |
| **Min Load** | pmin | Operate at minimum output for the full min up time, shut down |

**Minimum up time enforcement:** If `ramp_up_periods + ramp_down_periods < min_up_time`, the unit holds at its target dispatch level for the remaining periods. This always extends the highest power phase, not the ramp.

**Ramp-down mechanics:** From the target level, the unit ramps down by `ramp_down_limit` MW/hr each period until it reaches `ramp_shutdown_limit`, then goes offline the next period.

**Key insight for peaking economics:** Startup cost is fixed regardless of online duration. Min Load produces the fewest MWh over which to amortize that cost, so it typically yields the *highest* $/MWh despite the cheapest variable rate. Max Power produces the most MWh and lowest $/MWh.

Two summary tables are produced:
1. **Cost per MWh** — the all-in economics of each case
2. **Total online duration** — context for how long the unit runs (more hours dilutes the fixed startup cost)

In [7]:
def build_peaking_profile(gen: dict, target_mw: float) -> dict:
    """
    Build the dispatch profile for a peaking unit commitment.

    Sequence: ramp up from 0 to target_mw, hold at target_mw (if needed
    to satisfy min_up_time), ramp back down to ramp_shutdown_limit, then 0.

    Minimum up time constraint: if ramp_up_periods + ramp_down_periods <
    min_up_time, hold_periods = min_up_time - ramp_up_periods - ramp_down_periods.
    The hold always extends the highest dispatch level (target_mw).

    Returns dict with profile (list of MW), per-phase period counts, and
    whether the hold was needed to satisfy min_up_time (extended=True).
    """
    pmin          = float(gen["power_output_minimum"])
    ramp_startup  = float(gen.get("ramp_startup_limit",  pmin))
    ramp_up_lim   = float(gen.get("ramp_up_limit",   target_mw))
    ramp_down_lim = float(gen.get("ramp_down_limit", target_mw))
    ramp_shutdown = float(gen.get("ramp_shutdown_limit", pmin))
    min_up_time   = int(gen.get("time_up_minimum", 1))

    # Ramp up: 0 → target_mw
    ramp_up: list[float] = []
    current = 0.0
    while current < target_mw - 1e-6:
        new = min(target_mw, ramp_startup if not ramp_up else current + ramp_up_lim)
        ramp_up.append(new)
        current = new

    # Ramp down: target_mw → ramp_shutdown_limit (then unit goes offline)
    ramp_down: list[float] = []
    current = target_mw
    while current > ramp_shutdown + 1e-6:
        new = max(ramp_shutdown, current - ramp_down_lim)
        ramp_down.append(new)
        current = new

    # Hold at target_mw to satisfy min_up_time
    hold_needed = max(0, min_up_time - len(ramp_up) - len(ramp_down))
    hold = [target_mw] * hold_needed

    profile = ramp_up + hold + ramp_down

    return {
        "profile":           profile,
        "ramp_up_periods":   len(ramp_up),
        "hold_periods":      len(hold),
        "ramp_down_periods": len(ramp_down),
        "total_online":      len(profile),
        "extended":          len(hold) > 0,
    }


def compute_peaking_case(gen: dict, startup_cost: float, target_mw: float) -> dict:
    """Compute all cost and duration metrics for one peaking (fixed x variable) case."""
    result = build_peaking_profile(gen, target_mw)
    profile = result["profile"]

    variable_cost = sum(piecewise_cost_per_hr(gen, mw) * PERIOD_HRS for mw in profile)
    energy_mwh    = sum(mw * PERIOD_HRS for mw in profile)
    gross_cost    = startup_cost + variable_cost

    return {
        "startup_cost":      startup_cost,
        "variable_cost":     variable_cost,
        "gross_cost":        gross_cost,
        "energy_mwh":        energy_mwh,
        "cost_per_mwh":      gross_cost / energy_mwh if energy_mwh > 0 else float("nan"),
        "pct_fixed":         startup_cost / gross_cost * 100 if gross_cost > 0 else 0.0,
        "pct_variable":      variable_cost / gross_cost * 100 if gross_cost > 0 else 100.0,
        "ramp_up_periods":   result["ramp_up_periods"],
        "hold_periods":      result["hold_periods"],
        "ramp_down_periods": result["ramp_down_periods"],
        "total_online":      result["total_online"],
        "extended":          result["extended"],
    }


# Self-test: 322_CT_5 — cold start, min_up_time=3, fast ramp
_g  = thermal_gens["322_CT_5"]
_mc = find_min_mc_segment(_g)
_su = get_startup_tiers(_g)["cold"]["cost"]
print("322_CT_5  pmin=22  pmax=55  ramp_startup=22  ramp_up=222  ramp_shutdown=22  min_up=3")
print()
for label, target in [("max_power", 55.0), ("min_mc", _mc["target_mw"]), ("min_load", 22.0)]:
    p = build_peaking_profile(_g, target)
    c = compute_peaking_case(_g, _su, target)
    print(f"  {label:<12}  target={target:.0f}MW  profile={[round(v) for v in p['profile']]}  "
          f"online={p['total_online']}hr  {c['energy_mwh']:.0f}MWh  "
          f"${c['gross_cost']:,.2f}  ${c['cost_per_mwh']:.2f}/MWh  "
          f"fixed={c['pct_fixed']:.1f}%  extended={p['extended']}")

322_CT_5  pmin=22  pmax=55  ramp_startup=22  ramp_up=222  ramp_shutdown=22  min_up=3

  max_power     target=55MW  profile=[22, 55, 22]  online=3hr  99MWh  $9,615.35  $97.12/MWh  fixed=58.9%  extended=False
  min_mc        target=33MW  profile=[22, 33, 22]  online=3hr  77MWh  $9,017.01  $117.10/MWh  fixed=62.8%  extended=False
  min_load      target=22MW  profile=[22, 22, 22]  online=3hr  66MWh  $8,760.33  $132.73/MWh  fixed=64.7%  extended=True


In [8]:
PEAKING_FIXED_LEVELS = ["hot", "warm", "cold"]
PEAKING_VAR_LEVELS   = ["max_power", "min_mc", "min_load"]

records_peak = []

for name, gen in thermal_gens.items():
    tiers   = get_startup_tiers(gen)
    mc_info = find_min_mc_segment(gen)
    pmax    = float(gen["power_output_maximum"])
    pmin    = float(gen["power_output_minimum"])

    startup_costs = {
        "hot":  tiers["hot"]["cost"]  if tiers["hot"]  is not None else None,
        "warm": tiers["warm"]["cost"] if tiers["warm"] is not None else None,
        "cold": tiers["cold"]["cost"] if tiers["cold"] is not None else None,
    }
    targets = {
        "max_power": pmax,
        "min_mc":    mc_info["target_mw"],
        "min_load":  pmin,
    }

    for fixed_level in PEAKING_FIXED_LEVELS:
        sc = startup_costs[fixed_level]
        for var_level in PEAKING_VAR_LEVELS:
            base = {
                "name":             name,
                "fuel_category":    gen.get("fuel_category", "Unknown"),
                "pmin":             pmin,
                "pmax":             pmax,
                "fixed_level":      fixed_level,
                "var_level":        var_level,
                "min_mc_per_mwh":   mc_info["mc_per_mwh"],
                "min_mc_target_mw": mc_info["target_mw"],
                "min_up_time":      int(gen.get("time_up_minimum", 1)),
            }
            if sc is None:
                base["applicable"] = False
                base.update({k: None for k in [
                    "startup_cost", "variable_cost", "gross_cost", "energy_mwh",
                    "cost_per_mwh", "pct_fixed", "pct_variable",
                    "ramp_up_periods", "hold_periods", "ramp_down_periods",
                    "total_online", "extended",
                ]})
            else:
                base["applicable"] = True
                base.update(compute_peaking_case(gen, sc, targets[var_level]))
            records_peak.append(base)

df_peak = pd.DataFrame(records_peak)
df_peak_valid = df_peak[df_peak["applicable"]].copy()

print(f"Peaking cases  total={len(df_peak)}  applicable={df_peak['applicable'].sum()}  "
      f"N/A={( ~df_peak['applicable']).sum()}")
print()
print("Average online hours by fuel category and variable level:")
display(
    df_peak_valid.groupby(["fuel_category", "var_level"])["total_online"]
    .mean().round(1)
    .unstack("var_level")[["max_power", "min_mc", "min_load"]]
)

Peaking cases  total=657  applicable=351  N/A=306

Average online hours by fuel category and variable level:


var_level,max_power,min_mc,min_load
fuel_category,,,
Coal,9.40,9.40,9.40
Gas CC,8.00,8.00,8.00
Gas CT,3.00,3.00,3.00
Nuclear,24.00,24.00,24.00
Oil CT,3.00,3.00,1.00
Oil ST,4.00,4.00,4.00


In [9]:
CASE_ORDER_PEAK = [
    ("hot",  "max_power"), ("hot",  "min_mc"), ("hot",  "min_load"),
    ("warm", "max_power"), ("warm", "min_mc"), ("warm", "min_load"),
    ("cold", "max_power"), ("cold", "min_mc"), ("cold", "min_load"),
]


def _make_peak_pivot(df, value_col):
    piv = df.pivot_table(
        index=["name", "fuel_category", "pmin", "pmax", "min_up_time",
               "min_mc_per_mwh", "min_mc_target_mw"],
        columns=["fixed_level", "var_level"],
        values=value_col,
        aggfunc="first",
    )
    piv = piv.reindex(columns=pd.MultiIndex.from_tuples(CASE_ORDER_PEAK))
    piv.columns = [f"{fl}_{vl}" for fl, vl in piv.columns]
    return piv.reset_index().sort_values(["fuel_category", "name"]).set_index(["name", "fuel_category"])


# ── Pivot 1: $/MWh
piv_cpmwh = _make_peak_pivot(df_peak, "cost_per_mwh")
cost_cols_p = [c for c in piv_cpmwh.columns if any(f in c for f in ["hot_", "warm_", "cold_"])]

display(
    piv_cpmwh.style
    .format({
        "pmin":             "{:,.1f} MW",
        "pmax":             "{:,.1f} MW",
        "min_up_time":      "{:,.0f} hr",
        "min_mc_per_mwh":   "${:,.2f}/MWh",
        "min_mc_target_mw": "{:,.1f} MW",
        **{c: lambda x: f"${x:,.2f}" if pd.notna(x) else "N/A" for c in cost_cols_p},
    })
    .background_gradient(subset=cost_cols_p, cmap="RdYlGn_r", axis=None)
    .set_caption(
        "Peaking Archetype — Cost per MWh ($/MWh)  "
        "[columns: hot/warm/cold start  x  max power / min MC / min load]"
    )
)

# ── Pivot 2: Total online duration
piv_online = _make_peak_pivot(df_peak, "total_online")
online_cols = [c for c in piv_online.columns if any(f in c for f in ["hot_", "warm_", "cold_"])]

display(
    piv_online.style
    .format({
        "pmin": "{:,.1f} MW", "pmax": "{:,.1f} MW", "min_up_time": "{:,.0f} hr",
        "min_mc_per_mwh": "${:,.2f}/MWh", "min_mc_target_mw": "{:,.1f} MW",
        **{c: lambda x: f"{x:,.0f} hr" if pd.notna(x) else "N/A" for c in online_cols},
    })
    .background_gradient(subset=online_cols, cmap="Blues", axis=None)
    .set_caption(
        "Peaking Archetype — Total Online Duration (hours)  "
        "[more hours = startup cost amortized over more MWh]"
    )
)

,,pmin,pmax,min_up_time,min_mc_per_mwh,min_mc_target_mw,hot_max_power,hot_min_mc,hot_min_load,warm_max_power,warm_min_mc,warm_min_load,cold_max_power,cold_min_mc,cold_min_load
name,fuel_category,,,,,,,,,,,,,,
101_STEAM_3,Coal,30.0 MW,76.0 MW,8 hr,$14.19/MWh,45.3 MW,$35.67,$45.73,$57.82,$41.74,$55.17,$70.87,$43.48,$57.86,$74.60
101_STEAM_4,Coal,30.0 MW,76.0 MW,8 hr,$14.19/MWh,45.3 MW,$35.67,$45.73,$57.82,$41.74,$55.17,$70.87,$43.48,$57.86,$74.60
102_STEAM_3,Coal,30.0 MW,76.0 MW,8 hr,$18.46/MWh,45.3 MW,$36.27,$44.35,$54.27,$42.34,$53.79,$67.32,$44.07,$56.48,$71.05
102_STEAM_4,Coal,30.0 MW,76.0 MW,8 hr,$18.46/MWh,45.3 MW,$36.27,$44.35,$54.27,$42.34,$53.79,$67.32,$44.07,$56.48,$71.05
115_STEAM_3,Coal,62.0 MW,155.0 MW,8 hr,$20.40/MWh,93.0 MW,$37.55,$44.52,$53.57,$38.65,$46.22,$55.90,$45.35,$56.57,$70.13
116_STEAM_1,Coal,62.0 MW,155.0 MW,8 hr,$19.69/MWh,93.0 MW,$38.47,$47.08,$57.36,$39.56,$48.78,$59.68,$46.26,$59.13,$73.92
123_STEAM_2,Coal,62.0 MW,155.0 MW,8 hr,$19.43/MWh,93.0 MW,$38.05,$43.52,$52.56,$39.14,$45.21,$54.88,$45.84,$55.57,$69.12
123_STEAM_3,Coal,140.0 MW,350.0 MW,24 hr,$19.98/MWh,210.0 MW,N/A,N/A,N/A,$26.01,$28.19,$31.96,$27.94,$31.33,$36.53
201_STEAM_3,Coal,30.0 MW,76.0 MW,8 hr,$22.19/MWh,45.3 MW,$39.34,$47.52,$57.23,$45.42,$56.95,$70.28,$47.15,$59.65,$74.01


,,pmin,pmax,min_up_time,min_mc_per_mwh,min_mc_target_mw,hot_max_power,hot_min_mc,hot_min_load,warm_max_power,warm_min_mc,warm_min_load,cold_max_power,cold_min_mc,cold_min_load
name,fuel_category,,,,,,,,,,,,,,
101_STEAM_3,Coal,30.0 MW,76.0 MW,8 hr,$14.19/MWh,45.3 MW,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr
101_STEAM_4,Coal,30.0 MW,76.0 MW,8 hr,$14.19/MWh,45.3 MW,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr
102_STEAM_3,Coal,30.0 MW,76.0 MW,8 hr,$18.46/MWh,45.3 MW,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr
102_STEAM_4,Coal,30.0 MW,76.0 MW,8 hr,$18.46/MWh,45.3 MW,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr
115_STEAM_3,Coal,62.0 MW,155.0 MW,8 hr,$20.40/MWh,93.0 MW,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr
116_STEAM_1,Coal,62.0 MW,155.0 MW,8 hr,$19.69/MWh,93.0 MW,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr
123_STEAM_2,Coal,62.0 MW,155.0 MW,8 hr,$19.43/MWh,93.0 MW,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr
123_STEAM_3,Coal,140.0 MW,350.0 MW,24 hr,$19.98/MWh,210.0 MW,N/A,N/A,N/A,24 hr,24 hr,24 hr,24 hr,24 hr,24 hr
201_STEAM_3,Coal,30.0 MW,76.0 MW,8 hr,$22.19/MWh,45.3 MW,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr,8 hr
